In [0]:
# Import required libraries
import requests
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

In [0]:
# Retrieve API key from Databricks Secret Scope (secure method)
API_KEY = dbutils.secrets.get(scope="api-keys", key="football-api-key")
if not API_KEY:
    raise Exception("API key not found in secret scope")
print("API key loaded")

API key loaded


In [0]:
# Configuration
BASE_URL = "https://api.football-data.org/v4"

# --- Project & schemas (separate from your other work) ---
PROJECT = "premier_league"  # change name if you want a different project
SCHEMA_BRONZE = f"workspace.{PROJECT}_bronze"
SCHEMA_SILVER = f"workspace.{PROJECT}_silver"   # reserved for later
SCHEMA_GOLD   = f"workspace.{PROJECT}_gold"     # reserved for later

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_GOLD}")
print(f"Schemas ready: {SCHEMA_BRONZE}, {SCHEMA_SILVER}, {SCHEMA_GOLD}")

Schemas ready: workspace.premier_league_bronze, workspace.premier_league_silver, workspace.premier_league_gold


In [0]:
# Data Range 
SEASONS = {
    "2022": "2022-23",
    "2023": "2023-24",
    "2024": "2024-25",
}

print("Configuration loaded")

Configuration loaded


In [0]:
# function to fetch the data Programmatically 
def fetch_standings(season_year: str):
    """
    Fetch Premier League standings for a specific season
    
    Args:
        season_year: Year code (e.g., '2023' for 2023-24 season)
    
    Returns:
        dict: API response with standings data
    """
    url = f"{BASE_URL}/competitions/PL/standings"
    headers = {"X-Auth-Token": API_KEY}
    params = {"season": season_year}
    print(f"Fetching standings for season {season_year}...")
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    print(f"Successfully fetched data for season {season_year}")
    return resp.json()



In [0]:

# Fucntiuon to save the data as csv becuase the data is as nested json 
def save_standings_to_table(api_response: dict, season_name: str, season_year: str):
    """
    Convert API standings data to CSV format using PySpark
    
    Args:
        api_response: JSON response from API
        season_name: Human-readable season name (e.g., '2024-25')
        season_year: Year code for filename
    """
    if not api_response:
        print(f"No data to save for {season_name}")
        return None

    standings_table = api_response["standings"][0]["table"]
    flattened = [{
        "position": r["position"],
        "team_id": r["team"]["id"],
        "team_name": r["team"]["name"],
        "team_short_name": r["team"]["shortName"],
        "team_tla": r["team"]["tla"],
        "played_games": r["playedGames"],
        "won": r["won"],
        "draw": r["draw"],
        "lost": r["lost"],
        "points": r["points"],
        "goals_for": r["goalsFor"],
        "goals_against": r["goalsAgainst"],
        "goal_difference": r["goalDifference"],
        "season": season_name,
        "ingestion_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    } for r in standings_table]

    df = spark.createDataFrame(flattened)
    print(f"\nSample data for {season_name}:")
    df.show(5, truncate=False)

    target_table = f"{SCHEMA_BRONZE}.pl_standings_{season_year}"
    (df.write
       .mode("overwrite")
       .format("delta")
       .saveAsTable(target_table))

    print(f"Saved to Databricks workspace table: {target_table}")
    print(f"  View in Catalog → workspace → {PROJECT}_bronze → pl_standings_{season_year}")
    return df

In [0]:
# Ingest standings for each season
all_dataframes = []

for season_year, season_name in SEASONS.items():
    print("\n" + "="*60)
    print(f"Processing Season: {season_name}")
    print("="*60)

    try:
        data = fetch_standings(season_year)
        if data is None:
            print(f"No data returned for {season_name} (skipping)")
            continue
        df = save_standings_to_table(data, season_name, season_year)
        if df is not None:
            all_dataframes.append(df)
    except Exception as e:
        print(f"Skipping {season_name} due to error: {e}")

print("\n" + "="*60)
print(" INGESTION COMPLETE")
print(f" Successfully ingested {len(all_dataframes)} seasons")
print("="*60)



Processing Season: 2022-23
Fetching standings for season 2022...
Skipping 2022-23 due to error: 403 Client Error:  for url: https://api.football-data.org/v4/competitions/PL/standings?season=2022

Processing Season: 2023-24
Fetching standings for season 2023...
Successfully fetched data for season 2023

Sample data for 2023-24:
+----+---------------+-------------+---------+-------------------+----+------------+------+--------+-------+-------+--------------------+---------------+--------+---+
|draw|goal_difference|goals_against|goals_for|ingestion_date     |lost|played_games|points|position|season |team_id|team_name           |team_short_name|team_tla|won|
+----+---------------+-------------+---------+-------------------+----+------------+------+--------+-------+-------+--------------------+---------------+--------+---+
|7   |62             |34           |96       |2025-10-27 03:49:24|3   |38          |91    |1       |2023-24|65     |Manchester City FC  |Man City       |MCI     |28 |
|5

In [0]:
#  connect to the table and load some sample data for the season 2023-24 and 2024-25
df = spark.read.table(f"{SCHEMA_BRONZE}.pl_standings_2023")
df.show()




+----+---------------+-------------+---------+-------------------+----+------------+------+--------+-------+-------+--------------------+---------------+--------+---+
|draw|goal_difference|goals_against|goals_for|     ingestion_date|lost|played_games|points|position| season|team_id|           team_name|team_short_name|team_tla|won|
+----+---------------+-------------+---------+-------------------+----+------------+------+--------+-------+-------+--------------------+---------------+--------+---+
|   7|             62|           34|       96|2025-10-27 03:49:24|   3|          38|    91|       1|2023-24|     65|  Manchester City FC|       Man City|     MCI| 28|
|   5|             62|           29|       91|2025-10-27 03:49:24|   5|          38|    89|       2|2023-24|     57|          Arsenal FC|        Arsenal|     ARS| 28|
|  10|             45|           41|       86|2025-10-27 03:49:24|   4|          38|    82|       3|2023-24|     64|        Liverpool FC|      Liverpool|     LIV| 24

In [0]:
import pandas 

df = spark.read.table(f"{SCHEMA_BRONZE}.pl_standings_2024").toPandas()
df.head()   


,draw,goal_difference,goals_against,goals_for,ingestion_date,lost,played_games,points,position,season,team_id,team_name,team_short_name,team_tla,won
0,9,45,41,86,2025-10-27 03:49:26,4,38,84,1,2024-25,64,Liverpool FC,Liverpool,LIV,25
1,14,35,34,69,2025-10-27 03:49:26,4,38,74,2,2024-25,57,Arsenal FC,Arsenal,ARS,20
2,8,28,44,72,2025-10-27 03:49:26,9,38,71,3,2024-25,65,Manchester City FC,Man City,MCI,21
3,9,21,43,64,2025-10-27 03:49:26,9,38,69,4,2024-25,61,Chelsea FC,Chelsea,CHE,20
4,6,21,47,68,2025-10-27 03:49:26,12,38,66,5,2024-25,67,Newcastle United FC,Newcastle,NEW,20


In [0]:
import pandas 

df = spark.read.table(f"{SCHEMA_BRONZE}.pl_standings_2022").toPandas()
df.head()   


,draw,goal_difference,goals_against,goals_for,ingestion_date,lost,played_games,points,position,season,team_id,team_name,team_short_name,team_tla,won
0,9,45,41,86,2025-10-27 03:49:26,4,38,84,1,2024-25,64,Liverpool FC,Liverpool,LIV,25
1,14,35,34,69,2025-10-27 03:49:26,4,38,74,2,2024-25,57,Arsenal FC,Arsenal,ARS,20
2,8,28,44,72,2025-10-27 03:49:26,9,38,71,3,2024-25,65,Manchester City FC,Man City,MCI,21
3,9,21,43,64,2025-10-27 03:49:26,9,38,69,4,2024-25,61,Chelsea FC,Chelsea,CHE,20
4,6,21,47,68,2025-10-27 03:49:26,12,38,66,5,2024-25,67,Newcastle United FC,Newcastle,NEW,20


In [0]:
# get the how many seasons are ingested 
count = spark.read.table(f"{SCHEMA_BRONZE}.pl_standings_2024").count()
print(count)
count = spark.read.table(f"{SCHEMA_BRONZE}.pl_standings_2023").count()
print(count)

20
20
